# Certification Officielle : Prêt à Dépenser (Projet 8)

Ce notebook constitue le registre de certification final du modèle.

**Objectifs :**
1. **Rigueur** : Alignement strict du preprocessing (clipping).
2. **Transparence** : Extraction de toutes les métriques de performance.
3. **Arbitrage** : Validation du coût métier et du pouvoir de tri.
4. **Traçabilité** : Archivage automatique dans MLflow.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import time
import mlflow
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, auc
from sklearn.model_selection import train_test_split

# Configuration MLflow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("Certification_Production")

MODEL_PATH = "../model/optuna_scoring_model.joblib"
DATA_PATH = "../internal_doc/Projet 6/Projet+Mise+en+prod+-+home-credit-default-risk/application_train.csv"

artefact = joblib.load(MODEL_PATH)
metrics = artefact["metrics"]
print(f"Modèle chargé : {MODEL_PATH}")

## 1. Preprocessing Strict (Alignement)

In [ ]:
def create_domain_features_strict(df):
    """Réplique exactement la logique d'entraînement (clipping 0.99)"""
    df = df.copy()
    df["DAYS_EMPLOYED"].replace(365243, np.nan, inplace=True)
    for col in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY"]:
        if col in df.columns:
            upper_limit = df[col].quantile(0.99)
            df[col] = df[col].clip(upper=upper_limit)
    
    df["DAYS_EMPLOYED_PERCENT"] = df["DAYS_EMPLOYED" ] / df["DAYS_BIRTH"]
    df["YEARS_EMPLOYED"] = df["DAYS_EMPLOYED"] / -365
    df["YEARS_BIRTH"] = df["DAYS_BIRTH"] / -365
    df["CREDIT_INCOME_PERCENT"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["ANNUITY_INCOME_PERCENT"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["CREDIT_TERM"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    return df

df = pd.read_csv(DATA_PATH)
df = create_domain_features_strict(df)
y = df["TARGET"]
X = df.drop(columns=["TARGET", "SK_ID_CURR"])
cat_columns = X.select_dtypes(include=["object"]).columns.tolist()
X = pd.get_dummies(X, columns=cat_columns, drop_first=True)
X = X.rename(columns=lambda x: re.sub('[^A-Za-z0-9_]+', '', x))

_, X_val, _, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_val_top = X_val[artefact["features"]]
X_val_imp = artefact["imputer"].transform(X_val_top)

## 2. Évaluation & Tracking MLflow

Extraction de l'ensemble des métriques de classification et de coût métier.

In [ ]:
start_time = time.time()
y_p = artefact["model"].predict_proba(X_val_imp)[:, 1]
end_time = time.time()

calc_time = end_time - start_time
threshold = metrics["best_threshold"]
y_pred = (y_p >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_val, y_pred).ravel()
actual_cost = (fn * 10) + (fp * 1)
auc_score = roc_auc_score(y_val, y_p)
report = classification_report(y_val, y_pred, output_dict=True)

with mlflow.start_run(run_name="Certification_Finale"):
    # Métriques de base
    mlflow.log_metric("TN", int(tn))
    mlflow.log_metric("FP", int(fp))
    mlflow.log_metric("FN", int(fn))
    mlflow.log_metric("TP", int(tp))
    
    # Métriques de classification
    mlflow.log_metric("roc_auc", float(auc_score))
    mlflow.log_metric("precision", float(report['macro avg']['precision']))
    mlflow.log_metric("recall", float(report['macro avg']['recall']))
    mlflow.log_metric("f1_score", float(report['macro avg']['f1-score']))
    
    # Métriques métier
    mlflow.log_metric("business_cost", float(actual_cost))
    mlflow.log_metric("inference_time", float(calc_time))
    
    print("### CLASSIFICATION REPORT ###")
    print(classification_report(y_val, y_pred))
    print(f"Coût métier final : {actual_cost} $")
    print(f"Temps de calcul : {calc_time:.4f} s")

## 3. Visualisations Bancaires

### Heatmap des Coûts Financiers
Cette vue pondère les erreurs par leur coût réel pour la banque (FN=10, FP=1).

In [ ]:
cost_matrix = np.array([[tn*0, fp*1], [fn*10, tp*0]])

plt.figure(figsize=(8, 6))
sns.heatmap(cost_matrix, annot=True, fmt='g', cmap='Reds', 
            xticklabels=['Prêt Accordé', 'Prêt Refusé'],
            yticklabels=['Solvable', 'Défaut'])
plt.title("Distribution du Risque Financier ($)")
plt.xlabel("Décision du Modèle")
plt.ylabel("État Réel du Client")
plt.show()

print("Interprétation : La heatmap montre l'exposition financière. Les défauts non détectés (FN) sont pondérés x10.")

### Courbe de Gains Cumulés
Indispensable pour démontrer le pouvoir de tri par rapport à un choix aléatoire.

In [ ]:
def plot_cumulative_gains(y_true, y_score):
    df_gains = pd.DataFrame({'true': y_true, 'score': y_score})
    df_gains = df_gains.sort_values(by='score', ascending=False)
    n_pos = y_true.sum()
    pct_pop = np.arange(1, len(y_true) + 1) / len(y_true)
    pct_gains = np.cumsum(df_gains['true']) / n_pos
    
    plt.figure(figsize=(8, 6))
    plt.plot(pct_pop, pct_gains, color='blue', lw=3, label='Modèle')
    plt.plot([0, 1], [0, 1], 'r--', label='Aléatoire')
    plt.title("Pouvoir de Tri : Gains Cumulés")
    plt.xlabel("Population testée (% la plus risquée)")
    plt.ylabel("Capture des défauts (%)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

plot_cumulative_gains(y_val, y_p)

print("Interprétation : Plus la courbe est bombée, plus le modèle capte rapidement les mauvais payeurs.")